# Top 3 Modelos — Análisis de Sentimientos IMDb

Este notebook consolida los **tres mejores modelos** seleccionados de tres notebooks de análisis de sentimientos sobre el dataset IMDb Movie Reviews.

| # | Modelo | Origen | Framework | Por qué está en el Top 3 |
|---|--------|--------|-----------|---------------------------|
| 1 | **CNN-BiLSTM** | `Microproyecto_RNN_IMDb.ipynb` | PyTorch | Combina extracción local de n-gramas (CNN) con dependencias largas (LSTM bidireccional); mejor trade-off velocidad/rendimiento del notebook 1 |
| 2 | **LSTM Bidireccional + GloVe** | `MicroproyectoIIprueba.ipynb` | PyTorch | LSTM bidireccional de 2 capas con embeddings preentrenados GloVe 200d; fine-tuning adaptado al dominio cinematográfico |
| 3 | **BiLSTM + Attention (Bahdanau) + GloVe** | `FeelingAnalysis_mafe.ipynb` | TensorFlow/Keras | El modelo más sofisticado: GloVe 200d + BiLSTM + mecanismo de atención aditiva interpretable; arquitectura más cercana al techo RNN en IMDb (~0.92) |

**Dataset:** IMDb Movie Ratings Sentiment Analysis  
**Tarea:** Clasificación binaria — Positivo (1) / Negativo (0)  
**Dataset:** ~40 000 reseñas balanceadas

## 1. Librerías y configuración global

In [ ]:
# ── Instalaciones (descomenta si es necesario) ─────────────────────────────
# !pip install torch pandas numpy scikit-learn matplotlib seaborn
# !pip install tensorflow gensim

from __future__ import annotations

import os
import re
import json
import math
import random
import zipfile
import urllib.request
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix,
)

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')

# ── Semillas de reproducibilidad ───────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo de cómputo: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# ── Paleta visual ─────────────────────────────────────────────────────────
PAL = {'Positiva': '#388697', 'Negativa': '#DB162F'}
BG  = '#f8f9fa'

## 2. Carga y preprocesamiento del dataset

El dataset IMDb contiene ~40 000 reseñas de películas balanceadas (50 % positivas / 50 % negativas).
Se eliminan duplicados, se limpia el HTML y se construye un vocabulario **solo desde el conjunto de entrenamiento** para evitar *data leakage*.

**Split:** 70 % train · 15 % validación · 15 % test (estratificado).

In [ ]:
# ── Cargar datos ──────────────────────────────────────────────────────────
# Ajusta FILE_PATH a la ubicación real de tu archivo
FILE_PATH = 'movie.csv'

df = pd.read_csv(FILE_PATH)
print(f'Shape original : {df.shape}')
print(f'Nulos          :\n{df.isnull().sum()}')

# Eliminar duplicados
antes = len(df)
df = df.drop_duplicates(subset='text').reset_index(drop=True)
print(f'Duplicados eliminados: {antes - len(df)}')
print(f'Distribución de clases:\n{df["label"].value_counts()}')

In [ ]:
# ── Limpieza de texto ─────────────────────────────────────────────────────
def clean_text(text: str) -> str:
    """Elimina HTML, pasa a minúsculas, limpia caracteres no alfabéticos."""
    text = str(text)
    text = re.sub(r'<.*?>', ' ', text)       # tags HTML
    text = text.lower()
    text = re.sub(r"[^a-z\s']", ' ', text)  # puntuación y números
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)

# Eliminar reseñas vacías tras limpieza
df = df[df['clean_text'].str.strip() != ''].reset_index(drop=True)
print(f'Filas finales: {len(df):,}')
df[['text', 'clean_text', 'label']].head(3)

In [ ]:
# ── EDA rápido ────────────────────────────────────────────────────────────
df['word_count'] = df['clean_text'].apply(lambda x: len(x.split()))
df['sent_label'] = df['label'].map({1: 'Positiva', 0: 'Negativa'})

print('--- Estadísticas de longitud por clase ---')
print(df.groupby('sent_label')['word_count'].describe().T)

fig, axes = plt.subplots(1, 2, figsize=(14, 4), facecolor=BG)

# Distribución de clases
vc = df['sent_label'].value_counts()
bars = axes[0].bar(vc.index, vc.values, color=[PAL[k] for k in vc.index], width=0.5)
for b in bars:
    axes[0].text(b.get_x() + b.get_width()/2, b.get_height() + 30,
                 f'{int(b.get_height()):,}', ha='center', fontweight='bold')
axes[0].set_title('Distribución de Clases', fontweight='bold')
axes[0].set_facecolor(BG); axes[0].spines[['top','right']].set_visible(False)

# Distribución de longitudes
for label, color in PAL.items():
    axes[1].hist(df[df['sent_label']==label]['word_count'], bins=40,
                 alpha=0.65, color=color, label=label, edgecolor='white')
axes[1].set_title('Longitud de reseñas (palabras)', fontweight='bold')
axes[1].legend(); axes[1].set_facecolor(BG)
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# ── División train / val / test ───────────────────────────────────────────
train_df, temp_df = train_test_split(
    df, test_size=0.30, stratify=df['label'], random_state=SEED
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, stratify=temp_df['label'], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Train : {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}')
print(f'Balance en test  : {test_df["label"].value_counts().to_dict()}')

In [ ]:
# ── Vocabulario y encoding (PyTorch) ──────────────────────────────────────
MAX_VOCAB = 20_000
MAX_LEN   = 256     # cubre percentil 75 de longitudes

def build_vocab(texts, max_size=MAX_VOCAB):
    """Construye vocabulario sobre el conjunto de entrenamiento (sin data leakage)."""
    words = []
    for text in texts:
        words.extend(text.split())
    freq  = Counter(words)
    vocab = {'<pad>': 0, '<unk>': 1}
    for word, _ in freq.most_common(max_size - 2):
        vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(train_df['clean_text'])
print(f'Tamaño del vocabulario: {len(vocab):,}')

def encode(text, vocab, max_len=MAX_LEN):
    tokens = text.split()
    idxs   = [vocab.get(tok, vocab['<unk>']) for tok in tokens]
    if len(idxs) < max_len:
        idxs += [vocab['<pad>']] * (max_len - len(idxs))
    else:
        idxs = idxs[:max_len]
    return idxs


class TextDataset(Dataset):
    def __init__(self, df, vocab, max_len=MAX_LEN):
        self.texts  = df['clean_text'].tolist()
        self.labels = df['label'].tolist()
        self.vocab  = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(encode(self.texts[idx], self.vocab, self.max_len), dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y


BATCH_SIZE = 64

train_ds = TextDataset(train_df, vocab)
val_ds   = TextDataset(val_df,   vocab)
test_ds  = TextDataset(test_df,  vocab)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE)

print('DataLoaders PyTorch listos ✓')

## 3. Funciones de entrenamiento y evaluación (PyTorch)

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    for x_batch, y_batch in loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(x_batch)
        loss    = criterion(outputs, y_batch)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # evita explosión de gradientes
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


def evaluate_full(model, loader, criterion, device):
    """Retorna loss + accuracy, precision, recall, F1."""
    model.eval()
    total_loss = 0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            outputs     = model(x_batch)
            total_loss += criterion(outputs, y_batch).item()
            all_preds.extend(outputs.argmax(dim=1).cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())
    avg_loss = total_loss / len(loader)
    acc  = accuracy_score(all_labels,  all_preds)
    prec = precision_score(all_labels, all_preds, zero_division=0)
    rec  = recall_score(all_labels,    all_preds, zero_division=0)
    f1   = f1_score(all_labels,        all_preds, zero_division=0)
    return avg_loss, acc, prec, rec, f1, all_preds, all_labels


def run_training(model, model_name, train_loader, val_loader,
                 epochs=20, patience=3, lr=3e-4):
    """Entrena con early stopping y ReduceLROnPlateau."""
    model.to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    criterion = nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=2, factor=0.5
    )
    history = {'train_loss': [], 'val_loss': [], 'val_acc': [],
               'val_precision': [], 'val_recall': [], 'val_f1': []}
    best_val_loss    = float('inf')
    best_model_state = None
    no_improve       = 0

    print(f"\n{'='*60}")
    print(f'  Entrenando: {model_name}')
    print(f"{'='*60}")

    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
        val_loss, acc, prec, rec, f1, _, _ = evaluate_full(model, val_loader, criterion, DEVICE)
        lr_before = optimizer.param_groups[0]['lr']
        scheduler.step(val_loss)
        lr_after  = optimizer.param_groups[0]['lr']

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(acc)
        history['val_precision'].append(prec)
        history['val_recall'].append(rec)
        history['val_f1'].append(f1)

        lr_tag = f' ↓lr={lr_after:.2e}' if lr_after < lr_before else ''
        print(f'Época {epoch:02d} | train_loss={train_loss:.4f} | '
              f'val_loss={val_loss:.4f} | acc={acc:.4f} | f1={f1:.4f}{lr_tag}')

        if val_loss < best_val_loss:
            best_val_loss    = val_loss
            best_model_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve       = 0
            print(f'           ✓ Mejor modelo guardado (val_loss={best_val_loss:.4f})')
        else:
            no_improve += 1
            print(f'           Sin mejora {no_improve}/{patience}')
            if no_improve >= patience:
                print(f'\nEarly stopping en época {epoch}.')
                break

    model.load_state_dict(best_model_state)
    print('✓ Mejor modelo restaurado')
    return history


def plot_training_curves(history, model_name):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor=BG)
    epochs = range(1, len(history['train_loss']) + 1)

    axes[0].plot(epochs, history['train_loss'], label='Train', color='#2d6a8a')
    axes[0].plot(epochs, history['val_loss'],   label='Val',   color='#e07b39')
    axes[0].set_title('Loss'); axes[0].legend()

    axes[1].plot(epochs, history['val_acc'], color='#388697')
    axes[1].set_title('Accuracy (val)'); axes[1].set_ylim(0.5, 1.0)

    axes[2].plot(epochs, history['val_f1'],        label='F1',        color='#6A0572')
    axes[2].plot(epochs, history['val_precision'], label='Precision', color='#388697', linestyle='--')
    axes[2].plot(epochs, history['val_recall'],    label='Recall',    color='#DB162F', linestyle='--')
    axes[2].set_title('F1 / Precision / Recall (val)'); axes[2].legend()

    for ax in axes:
        ax.set_facecolor(BG); ax.spines[['top','right']].set_visible(False)
        ax.set_xlabel('Época')

    fig.suptitle(f'Curvas de entrenamiento — {model_name}', fontweight='bold')
    plt.tight_layout()
    plt.show()

---
## 4. Modelo 1 — CNN-BiLSTM

**Origen:** `Microproyecto_RNN_IMDb.ipynb`  
**Framework:** PyTorch

### Arquitectura
```
Embedding → Conv1d (kernel=5) → ReLU → BiLSTM (1 capa) → FC (hidden→hidden→1)
```

| Capa | Detalle | Propósito |
|------|---------|----------|
| `Embedding` | vocab_size × 256 | Representación densa de tokens |
| `Conv1d` | kernel=5, out=256 | Captura n-gramas locales (patrones de 5 palabras) |
| `BiLSTM` | 1 capa, hidden=256 | Dependencias secuenciales de largo alcance |
| `FC` | 512 → 256 → ReLU → 1 | Clasificador con regularización |

**Por qué es uno de los mejores:** La CNN actúa como un extractor de características locales (n-gramas de palabras) antes de pasarlos al LSTM, que captura el contexto secuencial. Esta combinación reduce el costo computacional del LSTM al procesarlo con representaciones ya comprimidas.

In [ ]:
class CNNBiLSTM(nn.Module):
    """
    CNN-BiLSTM para clasificación de sentimientos.
    Conv1d extrae n-gramas locales; BiLSTM captura dependencias de largo alcance.
    """
    def __init__(self, vocab_size, embed_dim=256, hidden_dim=256,
                 output_dim=2, dropout=0.35, kernel_size=5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.conv = nn.Conv1d(
            in_channels=embed_dim,
            out_channels=embed_dim,
            kernel_size=kernel_size,
            padding=kernel_size // 2,   # padding='same'
        )
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        # x: (batch, seq_len)
        embedded  = self.embedding(x)                    # (batch, seq, embed)
        conv_in   = embedded.transpose(1, 2)             # (batch, embed, seq)  ← Conv1d espera channels-first
        conv_out  = torch.relu(self.conv(conv_in))       # (batch, embed, seq)
        lstm_in   = self.dropout(conv_out.transpose(1, 2))  # (batch, seq, embed)
        _, (h, _) = self.lstm(lstm_in)
        # h: (2, batch, hidden) — 2 = forward + backward
        features  = torch.cat([h[0], h[1]], dim=1)      # (batch, hidden*2)
        return self.fc(self.dropout(features))


model_cnn_bilstm = CNNBiLSTM(vocab_size=len(vocab)).to(DEVICE)
print(model_cnn_bilstm)
total_params = sum(p.numel() for p in model_cnn_bilstm.parameters() if p.requires_grad)
print(f'\nParámetros entrenables: {total_params:,}')

In [ ]:
# ── Entrenamiento CNN-BiLSTM ───────────────────────────────────────────────
history_cnn_bilstm = run_training(
    model_cnn_bilstm, 'CNN-BiLSTM',
    train_loader, val_loader,
    epochs=20, patience=3, lr=2e-3
)
plot_training_curves(history_cnn_bilstm, 'CNN-BiLSTM')

---
## 5. Modelo 2 — LSTM Bidireccional + GloVe

**Origen:** `MicroproyectoIIprueba.ipynb`  
**Framework:** PyTorch

### Arquitectura
```
GloVe Embedding (200d, fine-tune) → Dropout → BiLSTM (2 capas) → Dropout → FC
```

| Capa | Detalle | Propósito |
|------|---------|----------|
| `GloVe Embedding` | 200d, fine-tuning activo | Semántica preentrenada sobre 6B tokens |
| `BiLSTM` | 2 capas, hidden=256, bidireccional | Jerarquía semántica + contexto global |
| `FC` | 512 → 2 | Clasificador final |

**Por qué es uno de los mejores:** Los embeddings GloVe aceleran la convergencia porque llegan con relaciones semánticas ya aprendidas ("good" ↔ "excellent"). Las 2 capas de BiLSTM apiladas capturan tanto patrones de bajo nivel como dependencias de largo alcance.

In [ ]:
# ── Descargar GloVe 6B 200d ───────────────────────────────────────────────
GLOVE_ZIP  = 'glove.6B.zip'
GLOVE_FILE = 'glove.6B.200d.txt'

if not os.path.exists(GLOVE_FILE):
    print('Descargando GloVe 6B 200d (~860 MB)... puede tardar varios minutos.')
    urllib.request.urlretrieve(
        'https://nlp.stanford.edu/data/glove.6B.zip', GLOVE_ZIP
    )
    print('Descomprimiendo...')
    with zipfile.ZipFile(GLOVE_ZIP, 'r') as z:
        z.extract(GLOVE_FILE)
    print('GloVe listo ✓')
else:
    print('GloVe ya existe, omitiendo descarga ✓')


def load_glove(glove_path, vocab, embed_dim=200):
    """
    Construye la matriz de embeddings alineada con el vocabulario.
    Palabras sin cobertura en GloVe se inicializan en ceros y se aprenden desde cero.
    """
    embedding_matrix = np.zeros((len(vocab), embed_dim), dtype=np.float32)
    encontradas = 0
    with open(glove_path, encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word   = values[0]
            if word in vocab:
                embedding_matrix[vocab[word]] = np.asarray(values[1:], dtype='float32')
                encontradas += 1
    cobertura = encontradas / len(vocab) * 100
    print(f'Cobertura GloVe: {encontradas:,}/{len(vocab):,} palabras ({cobertura:.1f}%)')
    return torch.tensor(embedding_matrix, dtype=torch.float)


glove_matrix = load_glove(GLOVE_FILE, vocab)

In [ ]:
class LSTMBidirGloVe(nn.Module):
    """
    LSTM Bidireccional de 2 capas con embeddings inicializados desde GloVe 200d.
    Fine-tuning activado: los embeddings se actualizan durante el entrenamiento.
    """
    def __init__(self, vocab_size, embedding_matrix,
                 embed_dim=200, hidden_dim=256, output_dim=2,
                 n_layers=2, dropout=0.3, freeze_embeddings=False):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding.weight              = nn.Parameter(embedding_matrix)
        self.embedding.weight.requires_grad = not freeze_embeddings

        self.emb_dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            dropout=dropout if n_layers > 1 else 0,
            bidirectional=True,
        )
        self.dropout    = nn.Dropout(dropout)
        self.fc         = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        embedded       = self.emb_dropout(self.embedding(x))   # (batch, seq, 200)
        _, (hidden, _) = self.lstm(embedded)
        hidden_fwd     = hidden[-2, :, :]   # última capa, forward
        hidden_bwd     = hidden[-1, :, :]   # última capa, backward
        hidden_cat     = torch.cat([hidden_fwd, hidden_bwd], dim=1)  # (batch, 512)
        return self.fc(self.dropout(hidden_cat))


model_lstm_glove = LSTMBidirGloVe(
    vocab_size=len(vocab),
    embedding_matrix=glove_matrix,
    freeze_embeddings=False,
).to(DEVICE)

print(model_lstm_glove)
total_params = sum(p.numel() for p in model_lstm_glove.parameters() if p.requires_grad)
print(f'\nParámetros entrenables: {total_params:,}')

In [ ]:
# ── Entrenamiento LSTM Bidireccional + GloVe ──────────────────────────────
history_lstm_glove = run_training(
    model_lstm_glove, 'LSTM Bidireccional + GloVe',
    train_loader, val_loader,
    epochs=20, patience=3, lr=3e-4
)
plot_training_curves(history_lstm_glove, 'LSTM Bidireccional + GloVe')

---
## 6. Modelo 3 — BiLSTM + Attention (Bahdanau) + GloVe

**Origen:** `FeelingAnalysis_mafe.ipynb`  
**Framework:** TensorFlow / Keras

### Arquitectura
```
GloVe Embedding (200d) → BiLSTM → Attention aditiva (Bahdanau) → Dense → Sigmoid
```

| Capa | Detalle | Propósito |
|------|---------|----------|
| `GloVe Embedding` | 200d, mask_zero=True, fine-tunable | Semántica preentrenada con soporte a masking de padding |
| `BiLSTM` | lstm_units=64 (128 total), return_sequences=True | Estado oculto en **cada paso** temporal |
| `AttentionLayer` | Bahdanau aditiva, units=64 | Pondera los pasos temporales según importancia; interpretable |
| `Dense(64) + Sigmoid` | Con Dropout | Clasificador final |

**Por qué es el más sofisticado:** La atención de Bahdanau aprende a "mirar" qué palabras son más relevantes para la decisión (palabras como *brilliant*, *terrible*, *not*). Esto supera el cuello de botella clásico del LSTM que solo usa el último estado oculto, especialmente en reseñas largas.

**Fórmula de atención:**
$$u_t = \tanh(W h_t + b) \quad \alpha_t = \text{softmax}(v^\top u_t) \quad c = \sum_t \alpha_t h_t$$

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM, Dense, Dropout, Layer
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import gensim.downloader as gensim_api

print(f'TensorFlow: {tf.__version__}')

# ── Reproducibilidad Keras ────────────────────────────────────────────────
tf.keras.utils.set_random_seed(SEED)

# ── Detección de GPU y mixed precision ───────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'GPU detectada: {gpus[0].name}')
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    tf.keras.mixed_precision.set_global_policy('mixed_float16')
    print(f'Mixed precision: {tf.keras.mixed_precision.global_policy().name}')
else:
    print('Sin GPU — entrenando en CPU (float32)')
    tf.keras.mixed_precision.set_global_policy('float32')

In [ ]:
# ── Tokenización Keras (para el modelo de TF) ────────────────────────────
MAX_WORDS_TF = 10_000
MAX_LEN_TF   = 180    # percentil ~80; ~40% más rápido que 300
BATCH_SIZE_TF = 256

X_text = df['clean_text']
y      = df['label'].values

# Re-hacer los splits sobre el df completo (mismo SEED → mismos índices)
X_train_text, X_temp_text, y_train_tf, y_temp_tf = train_test_split(
    X_text, y, test_size=0.30, random_state=SEED, stratify=y
)
X_val_text, X_test_text, y_val_tf, y_test_tf = train_test_split(
    X_temp_text, y_temp_tf, test_size=0.50, random_state=SEED, stratify=y_temp_tf
)

tokenizer_tf = Tokenizer(num_words=MAX_WORDS_TF, oov_token='<OOV>')
tokenizer_tf.fit_on_texts(X_train_text)   # solo train — sin data leakage

X_train_seq = tokenizer_tf.texts_to_sequences(X_train_text)
X_val_seq   = tokenizer_tf.texts_to_sequences(X_val_text)
X_test_seq  = tokenizer_tf.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN_TF, padding='post', truncating='post')
X_val_pad   = pad_sequences(X_val_seq,   maxlen=MAX_LEN_TF, padding='post', truncating='post')
X_test_pad  = pad_sequences(X_test_seq,  maxlen=MAX_LEN_TF, padding='post', truncating='post')

vocab_size_tf = min(MAX_WORDS_TF, len(tokenizer_tf.word_index) + 1)
print(f'vocab_size_tf: {vocab_size_tf:,}  |  max_len_tf: {MAX_LEN_TF}')
print(f'Shapes — train: {X_train_pad.shape}  val: {X_val_pad.shape}  test: {X_test_pad.shape}')

# ── Pipelines tf.data ─────────────────────────────────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

train_pipeline_tf = (
    tf.data.Dataset.from_tensor_slices((X_train_pad, y_train_tf))
    .cache().shuffle(len(X_train_pad), seed=SEED).batch(BATCH_SIZE_TF).prefetch(AUTOTUNE)
)
val_pipeline_tf = (
    tf.data.Dataset.from_tensor_slices((X_val_pad, y_val_tf))
    .cache().batch(BATCH_SIZE_TF).prefetch(AUTOTUNE)
)
test_pipeline_tf = (
    tf.data.Dataset.from_tensor_slices((X_test_pad, y_test_tf))
    .cache().batch(BATCH_SIZE_TF).prefetch(AUTOTUNE)
)
print('Pipelines tf.data listos ✓')

In [ ]:
# ── Matriz de embeddings GloVe para Keras ────────────────────────────────
EMBED_DIM_TF = 200

print('Cargando GloVe glove-wiki-gigaword-200 vía gensim...')
glove_gensim = gensim_api.load('glove-wiki-gigaword-200')
print(f'Vectores disponibles: {len(glove_gensim):,}  |  Dimensión: {glove_gensim.vector_size}')

embedding_matrix_tf = np.zeros((vocab_size_tf, EMBED_DIM_TF), dtype=np.float32)
covered = 0
for word, idx in tokenizer_tf.word_index.items():
    if idx >= vocab_size_tf:
        continue
    if word in glove_gensim:
        embedding_matrix_tf[idx] = glove_gensim[word]
        covered += 1

total_tf = vocab_size_tf - 1
print(f'Cobertura GloVe (Keras): {covered:,}/{total_tf:,} = {covered/total_tf*100:.1f}%')
print(f'Shape matriz de embeddings: {embedding_matrix_tf.shape}')

In [ ]:
# ── Capa de Attention aditiva (Bahdanau) ─────────────────────────────────
class AttentionLayer(Layer):
    """
    Self-attention aditiva (Bahdanau) para colapsar (batch, time, feat) → (batch, feat).
    Aprende W, b, v y respeta la máscara de padding propagada desde el Embedding.
    Devuelve (context, alpha) — context es el resumen ponderado, alpha son los pesos
    de atención por paso temporal (útiles para visualización e interpretabilidad).

    Fórmula:
        u_t = tanh(W·h_t + b)        → proyección no lineal
        score_t = v^T · u_t           → escalar por paso temporal
        alpha_t = softmax(score_t)    → normalización con masking de padding
        c = Σ alpha_t · h_t           → vector de contexto final
    """
    def __init__(self, units=64, **kwargs):
        super().__init__(**kwargs)
        self.units = units
        self.supports_masking = True

    def build(self, input_shape):
        feat_dim = input_shape[-1]
        self.W = self.add_weight('W', shape=(feat_dim, self.units),
                                 initializer='glorot_uniform', trainable=True)
        self.b = self.add_weight('b', shape=(self.units,),
                                 initializer='zeros', trainable=True)
        self.v = self.add_weight('v', shape=(self.units, 1),
                                 initializer='glorot_uniform', trainable=True)
        super().build(input_shape)

    def call(self, inputs, mask=None):
        u      = tf.tanh(tf.matmul(inputs, self.W) + self.b)       # (batch, time, units)
        scores = tf.squeeze(tf.matmul(u, self.v), axis=-1)         # (batch, time)
        if mask is not None:
            mask_f = tf.cast(mask, scores.dtype)
            scores = scores + (1.0 - mask_f) * -1e9   # -inf en posiciones de padding
        alpha   = tf.nn.softmax(scores, axis=-1)                   # (batch, time)
        context = tf.reduce_sum(inputs * tf.expand_dims(alpha, -1), axis=1)  # (batch, feat)
        return context, alpha

    def compute_mask(self, inputs, mask=None):
        return None   # tras colapsar el tiempo ya no se propaga máscara

    def get_config(self):
        cfg = super().get_config()
        cfg['units'] = self.units
        return cfg


print('AttentionLayer (Bahdanau) definida ✓')

In [ ]:
# ── Construcción del modelo BiLSTM + Attention + GloVe ────────────────────
def build_bilstm_attention(
    vocab_size, max_len, embedding_matrix,
    embedding_dim=200, lstm_units=64, attn_units=64,
    dropout1=0.3, dropout2=0.2,
    embedding_trainable=True,
):
    """
    GloVe + BiLSTM + Attention aditiva (Bahdanau) + clasificador.
    Retorna (model, attention_model) — el segundo expone los pesos de atención.
    """
    inputs = Input(shape=(max_len,), dtype='int32', name='tokens')

    x = Embedding(
        input_dim=vocab_size, output_dim=embedding_dim,
        embeddings_initializer=tf.keras.initializers.Constant(embedding_matrix),
        mask_zero=True,       # propaga máscara de padding a la AttentionLayer
        trainable=embedding_trainable,
        name='embedding',
    )(inputs)

    x = Bidirectional(
        LSTM(lstm_units, return_sequences=True, dropout=0.3, recurrent_dropout=0.0),
        name='bilstm',
    )(x)
    # x: (batch, max_len, lstm_units*2)  — estados en CADA paso temporal

    context, alpha = AttentionLayer(units=attn_units, name='attention')(x)
    # context: (batch, lstm_units*2)  |  alpha: (batch, max_len)

    h = Dropout(dropout1, name='dropout_1')(context)
    h = Dense(64, activation='relu', name='dense_1')(h)
    h = Dropout(dropout2, name='dropout_2')(h)
    output = Dense(1, activation='sigmoid', dtype='float32', name='output')(h)

    model      = Model(inputs, output, name='BiLSTM_Attention_GloVe')
    attn_model = Model(inputs, [output, alpha], name='BiLSTM_Attention_GloVe_interpretable')
    return model, attn_model


model_attn, attn_model = build_bilstm_attention(
    vocab_size=vocab_size_tf,
    max_len=MAX_LEN_TF,
    embedding_matrix=embedding_matrix_tf,
)
model_attn.summary()

In [ ]:
# ── Compilar y entrenar BiLSTM + Attention + GloVe ───────────────────────
model_attn.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')],
)

callbacks_attn = [
    EarlyStopping(
        monitor='val_loss', patience=4,
        restore_best_weights=True, verbose=1,
        start_from_epoch=5,
    ),
    ReduceLROnPlateau(
        monitor='val_loss', factor=0.5, patience=2,
        min_lr=1e-6, verbose=1,
    ),
]

history_attn = model_attn.fit(
    train_pipeline_tf,
    validation_data=val_pipeline_tf,
    epochs=25,
    callbacks=callbacks_attn,
    verbose=1,
)

In [ ]:
# ── Curvas de entrenamiento (Keras) ───────────────────────────────────────
hist = history_attn.history
epochs_run = range(1, len(hist['loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor=BG)

axes[0].plot(epochs_run, hist['loss'],     label='Train', color='#2d6a8a')
axes[0].plot(epochs_run, hist['val_loss'], label='Val',   color='#e07b39')
axes[0].set_title('Loss'); axes[0].legend()

axes[1].plot(epochs_run, hist['accuracy'],     label='Train', color='#2d6a8a')
axes[1].plot(epochs_run, hist['val_accuracy'], label='Val',   color='#e07b39')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].set_ylim(0.5, 1.0)

axes[2].plot(epochs_run, hist['auc'],     label='Train AUC', color='#6A0572')
axes[2].plot(epochs_run, hist['val_auc'], label='Val AUC',   color='#DB162F', linestyle='--')
axes[2].set_title('AUC'); axes[2].legend(); axes[2].set_ylim(0.5, 1.0)

for ax in axes:
    ax.set_facecolor(BG); ax.spines[['top','right']].set_visible(False)
    ax.set_xlabel('Época')

fig.suptitle('Curvas de entrenamiento — BiLSTM + Attention + GloVe', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 7. Evaluación final en test — los 3 modelos

In [ ]:
# ── Evaluación PyTorch (Modelos 1 y 2) ────────────────────────────────────
criterion_eval = nn.CrossEntropyLoss()

results_summary = []

for model_obj, name in [
    (model_cnn_bilstm, 'CNN-BiLSTM'),
    (model_lstm_glove, 'LSTM Bidir + GloVe'),
]:
    _, acc, prec, rec, f1, preds, labels = evaluate_full(
        model_obj, test_loader, criterion_eval, DEVICE
    )
    results_summary.append({
        'Modelo': name, 'Framework': 'PyTorch',
        'Accuracy': round(acc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
    })
    print(f'\n=== {name} ===')
    print(classification_report(labels, preds, target_names=['Negativa', 'Positiva']))

# ── Evaluación TensorFlow (Modelo 3) ──────────────────────────────────────
test_loss_attn, test_acc_attn, test_auc_attn = model_attn.evaluate(test_pipeline_tf, verbose=0)
preds_attn   = (model_attn.predict(test_pipeline_tf, verbose=0) >= 0.5).astype(int).flatten()
labels_attn  = y_test_tf

prec_attn = precision_score(labels_attn, preds_attn, zero_division=0)
rec_attn  = recall_score(labels_attn,    preds_attn, zero_division=0)
f1_attn   = f1_score(labels_attn,        preds_attn, zero_division=0)

results_summary.append({
    'Modelo': 'BiLSTM + Attention + GloVe', 'Framework': 'TensorFlow/Keras',
    'Accuracy': round(test_acc_attn, 4),
    'Precision': round(prec_attn, 4),
    'Recall': round(rec_attn, 4),
    'F1-Score': round(f1_attn, 4),
})
print(f'\n=== BiLSTM + Attention + GloVe ===')
print(classification_report(labels_attn, preds_attn, target_names=['Negativa', 'Positiva']))

In [ ]:
# ── Tabla resumen ─────────────────────────────────────────────────────────
df_results = pd.DataFrame(results_summary).sort_values('F1-Score', ascending=False).reset_index(drop=True)
df_results.index += 1   # ranking 1, 2, 3
print('\n' + '='*65)
print('        RANKING FINAL — TOP 3 MODELOS (test set)')
print('='*65)
print(df_results.to_string())

In [ ]:
# ── Gráfico comparativo ───────────────────────────────────────────────────
metrics_cols = ['Accuracy', 'Precision', 'Recall', 'F1-Score']
models_names = df_results['Modelo'].tolist()
colors       = ['#388697', '#6A0572', '#DB162F']

x      = np.arange(len(metrics_cols))
width  = 0.25
fig, ax = plt.subplots(figsize=(12, 5), facecolor=BG)

for i, (_, row) in enumerate(df_results.iterrows()):
    vals = [row[m] for m in metrics_cols]
    bars = ax.bar(x + i * width, vals, width, label=row['Modelo'],
                  color=colors[i], alpha=0.85, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x + width)
ax.set_xticklabels(metrics_cols, fontweight='bold')
ax.set_ylim(0.80, 1.01)
ax.set_ylabel('Score')
ax.set_title('Comparación de métricas — Top 3 Modelos (test set)', fontweight='bold')
ax.legend(loc='lower right')
ax.set_facecolor(BG)
ax.spines[['top','right']].set_visible(False)
ax.axhline(0.90, color='gray', linestyle='--', linewidth=0.8, alpha=0.6, label='Referencia 0.90')

plt.tight_layout()
plt.show()

In [ ]:
# ── Matrices de confusión ─────────────────────────────────────────────────
all_preds_list  = []
all_labels_list = []
all_names       = []

# Recolectar predicciones PyTorch
for model_obj, name in [
    (model_cnn_bilstm, 'CNN-BiLSTM'),
    (model_lstm_glove, 'LSTM Bidir + GloVe'),
]:
    _, _, _, _, _, preds, labels = evaluate_full(model_obj, test_loader, criterion_eval, DEVICE)
    all_preds_list.append(preds)
    all_labels_list.append(labels)
    all_names.append(name)

# TF model
all_preds_list.append(preds_attn.tolist())
all_labels_list.append(labels_attn.tolist())
all_names.append('BiLSTM + Attention + GloVe')

fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor=BG)
for ax, preds, labels, name in zip(axes, all_preds_list, all_labels_list, all_names):
    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['Neg', 'Pos'], yticklabels=['Neg', 'Pos'])
    ax.set_title(name, fontweight='bold')
    ax.set_xlabel('Predicción'); ax.set_ylabel('Real')

fig.suptitle('Matrices de Confusión — Top 3 Modelos', fontweight='bold')
plt.tight_layout()
plt.show()

## 8. Visualización de atención — interpretabilidad del Modelo 3

El modelo BiLSTM + Attention aprende qué palabras "mirar" para decidir el sentimiento.
Podemos visualizar los pesos α por token para entender la decisión del modelo.

In [ ]:
# ── Visualizar atención en reseñas de ejemplo ─────────────────────────────
sample_reviews = [
    "This film was absolutely brilliant and the acting was superb.",
    "What a terrible waste of time. The plot made no sense at all.",
    "Not bad, but not great either. Some good moments, many boring ones.",
]

sample_seqs = tokenizer_tf.texts_to_sequences(
    [clean_text(r) for r in sample_reviews]
)
sample_pad  = pad_sequences(sample_seqs, maxlen=MAX_LEN_TF, padding='post', truncating='post')

pred_scores, alphas = attn_model.predict(sample_pad, verbose=0)

idx2word = {v: k for k, v in tokenizer_tf.word_index.items()}

fig, axes = plt.subplots(3, 1, figsize=(16, 9), facecolor=BG)

for i, (review, alpha, score) in enumerate(zip(sample_reviews, alphas, pred_scores)):
    tokens   = clean_text(review).split()
    token_ids = sample_seqs[i]
    words    = [idx2word.get(tid, '<OOV>') for tid in token_ids if tid != 0]
    weights  = alpha[: len(words)]

    sentiment = 'POSITIVA ✓' if score[0] >= 0.5 else 'NEGATIVA ✗'
    conf      = abs(score[0] - 0.5) * 200

    # Top 10 palabras por peso de atención
    top_n = min(10, len(words))
    sorted_idx = np.argsort(weights)[::-1][:top_n]
    top_words  = [words[j]   for j in sorted_idx]
    top_ws     = [weights[j] for j in sorted_idx]

    colors_bar = ['#DB162F' if w > np.mean(weights) else '#388697' for w in top_ws]
    axes[i].barh(range(top_n), top_ws, color=colors_bar, edgecolor='white')
    axes[i].set_yticks(range(top_n))
    axes[i].set_yticklabels(top_words, fontsize=9)
    axes[i].set_title(
        f'Reseña {i+1}: "{review[:60]}..." → {sentiment} (confianza: {conf:.1f}%)',
        fontsize=9, fontweight='bold'
    )
    axes[i].set_facecolor(BG)
    axes[i].spines[['top','right']].set_visible(False)
    axes[i].invert_yaxis()

fig.suptitle('Pesos de atención — palabras más relevantes por reseña', fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Inferencia — predicción en texto nuevo

Función unificada para predecir con cualquiera de los tres modelos.

In [ ]:
def predict_pytorch(model, text, vocab, max_len=MAX_LEN, device=DEVICE):
    """Predice sentimiento con un modelo PyTorch."""
    model.eval()
    clean = clean_text(text)
    ids   = torch.tensor([encode(clean, vocab, max_len)], dtype=torch.long).to(device)
    with torch.no_grad():
        logits = model(ids)
        probs  = torch.softmax(logits, dim=1)[0]
    label = 'POSITIVA ✓' if probs[1].item() >= 0.5 else 'NEGATIVA ✗'
    return label, probs[1].item()


def predict_keras(model, text, tokenizer, max_len=MAX_LEN_TF):
    """Predice sentimiento con el modelo TensorFlow/Keras."""
    clean = clean_text(text)
    seq   = tokenizer.texts_to_sequences([clean])
    pad   = pad_sequences(seq, maxlen=max_len, padding='post', truncating='post')
    score = model.predict(pad, verbose=0)[0][0]
    label = 'POSITIVA ✓' if score >= 0.5 else 'NEGATIVA ✗'
    return label, float(score)


# ── Prueba con nuevas reseñas ─────────────────────────────────────────────
test_texts_new = [
    "An absolute masterpiece. The direction and storytelling were breathtaking.",
    "I fell asleep halfway through. Completely boring and predictable.",
    "Not the worst film I've seen, but definitely not worth the ticket price.",
]

print(f"{'Reseña':<55} {'CNN-BiLSTM':<22} {'LSTM+GloVe':<22} {'Attention+GloVe'}")
print('-' * 125)

for text in test_texts_new:
    lbl1, sc1 = predict_pytorch(model_cnn_bilstm, text, vocab)
    lbl2, sc2 = predict_pytorch(model_lstm_glove, text, vocab)
    lbl3, sc3 = predict_keras(model_attn, text, tokenizer_tf)
    short = text[:52] + '...' if len(text) > 55 else text
    print(f'{short:<55} {lbl1} ({sc1:.2f})    {lbl2} ({sc2:.2f})    {lbl3} ({sc3:.2f})')

---
## 10. Resumen final y conclusiones

### Ranking de modelos (por F1-Score en test)

| Rank | Modelo | Framework | Fortaleza principal |
|------|--------|-----------|---------------------|
| 🥇 | **BiLSTM + Attention + GloVe** | TF/Keras | Arquitectura más expresiva: GloVe + mecanismo de atención interpretable; el más cercano al techo RNN en IMDb |
| 🥈 | **LSTM Bidireccional + GloVe** | PyTorch | Excelente generalización: 2 capas apiladas + fine-tuning de GloVe; balance ideal capacidad/costo |
| 🥉 | **CNN-BiLSTM** | PyTorch | Mayor velocidad de entrenamiento; la CNN extrae n-gramas locales antes del LSTM, reduciendo su carga computacional |

### Lecciones clave

1. **Embeddings preentrenados (GloVe)** son la mejora individual más impactante: aceleran la convergencia y reducen el overfitting.
2. **La atención supera el cuello de botella** del LSTM clásico en reseñas largas al ponderar todos los estados ocultos, no solo el último.
3. **CNN como preprocesador** del LSTM es una solución eficiente: captura patrones locales (n-gramas) y los pasa comprimidos al LSTM.
4. El **techo RNN puro en IMDb es ~0.92** de accuracy. Para superar ese umbral se requieren Transformers (BERT, RoBERTa).

### Próximos pasos sugeridos

- Probar **BERT fine-tuning** (`distilbert-base-uncased`) para superar el techo RNN.
- Añadir **stacked BiLSTM** (2 capas) al modelo de Atención.
- Explorar **búsqueda de hiperparámetros** (Optuna / KerasTuner) sobre el modelo de Atención.
- Implementar **ensemble** de los 3 modelos (promedio de probabilidades) para reducir varianza.